> **SUPERSEDED for interactive use by `notebooks/V1_Colab.ipynb`** — Colab gives every notebook a fresh VM, so the environment (deps, Drive mount, repo clone, env vars) does NOT persist between notebooks. Run everything in one: `notebooks/V1_Colab.ipynb`.
# 03 — Evaluate

Runs `scripts/colab_eval.sh` (pull + evaluate latest checkpoint on `$DATA_DIR/test.parquet`), renders `results.csv` as a markdown table, and plots RF vs n_tips / RF vs seq_len. Figures are saved to `$RESULTS_DIR`.

In [ ]:
import os, subprocess, sys, tempfile, pathlib
REPO_DIR = os.path.abspath(os.getcwd())
sys.path.insert(0, REPO_DIR)
DRIVE_SKIP = os.environ.get("COLAB_DRIVE_SKIP", "0") == "1"
if not os.environ.get("COLAB_DRIVE"):
    COLAB_DRIVE = ("/content/drive/MyDrive/ssm-phylo" if not DRIVE_SKIP
                   else tempfile.mkdtemp(prefix="ssm_drive_"))
    os.environ.update(
        COLAB_DRIVE=COLAB_DRIVE,
        DATA_DIR=f"{COLAB_DRIVE}/data",
        LOCAL_CKPT_DIR="/content/ckpts" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_ckpts",
        LOCAL_DATA_DIR="/content/data" if not DRIVE_SKIP else f"{COLAB_DRIVE}/local_data",
        CKPT_DIR=f"{COLAB_DRIVE}/checkpoints",
        RESULTS_DIR=f"{COLAB_DRIVE}/results",
    )
    for d in [os.environ["LOCAL_DATA_DIR"], os.environ["LOCAL_CKPT_DIR"],
              os.environ["DATA_DIR"], os.environ["CKPT_DIR"], os.environ["RESULTS_DIR"]]:
        pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", os.environ.get("DATA_DIR"))
print("CKPT_DIR:", os.environ.get("CKPT_DIR"))
print("RESULTS_DIR:", os.environ.get("RESULTS_DIR"))


In [ ]:
#@title Run evaluation
max_alignments = 200  #@param {type:"integer"}
cmd = ["bash", f"{REPO_DIR}/scripts/colab_eval.sh"]
if DRIVE_SKIP:
    import glob
    ckpts = sorted(glob.glob(os.path.join(os.environ["LOCAL_CKPT_DIR"], "*.pt")))
    if not ckpts:
        raise SystemExit("no checkpoints found — run 02_train first")
    test_p = os.path.join(os.environ["DATA_DIR"], "test.parquet")
    cmd = [sys.executable, "-m", "ssm_phylo.evaluate",
           "--checkpoint", ckpts[-1],
           "--test-parquet", test_p,
           "--out-dir", os.path.join(os.environ["RESULTS_DIR"], "eval_v1"),
           "--max-alignments", str(max_alignments)]
proc = subprocess.run(cmd, env=os.environ, cwd=REPO_DIR, text=True)
print("evaluate exit:", proc.returncode)

In [ ]:
#@title results.csv as markdown
import csv, os
out_dir = os.path.join(os.environ["RESULTS_DIR"], "eval_v1")
csv_path = os.path.join(out_dir, "results.csv")
if os.path.exists(csv_path):
    rows = list(csv.DictReader(open(csv_path)))
    print(f"| {' | '.join(rows[0].keys())} |")
    print(f"|{'---|' * len(rows[0])}")
    for r in rows[:20]:
        print(f"| {' | '.join(r.values())} |")
    print(f"... {len(rows)} rows total")
else:
    print(f"no results.csv at {csv_path}")

In [ ]:
#@title RF vs n_tips and RF vs seq_len (saved to $RESULTS_DIR)
import csv, os
import matplotlib.pyplot as plt

csv_path = os.path.join(os.environ["RESULTS_DIR"], "eval_v1", "results.csv")
if os.path.exists(csv_path):
    rows = list(csv.DictReader(open(csv_path)))
    x1 = [int(r["n_tips"]) for r in rows]
    y = [float(r["rf_pred"]) for r in rows]
    x2 = [float(r["seq_len"]) for r in rows]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
    ax1.scatter(x1, y, alpha=0.6)
    ax1.set_xlabel("n_tips"); ax1.set_ylabel("RF (predicted vs true)")
    ax1.set_title("RF vs n_tips")
    ax2.scatter(x2, y, alpha=0.6)
    ax2.set_xlabel("mean seq length"); ax2.set_ylabel("RF")
    ax2.set_title("RF vs seq_len")
    plt.tight_layout()
    fig_path = os.path.join(os.environ["RESULTS_DIR"], "eval_v1", "rf_plots.png")
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print("saved:", fig_path)
else:
    print("no results.csv yet — run the evaluate cell first")